# 从零复现 MobileNet 与 DenseNet：轻量卷积、倒残差和密集连接

这份 Notebook 不调用 `torchvision.models` 或 `timm`。我们只用基础张量算子和 `torch.nn` 的卷积、归一化、激活层，显式实现 `DepthwiseSeparableConv`、`MobileNetV1`、`InvertedResidual`、`MobileNetV2Tiny`、`DenseLayer`、`DenseBlock`、`Transition` 与 `DenseNetTiny.forward`。

重点不是把论文结构抄成一串层，而是回答工程复现时真正容易出错的问题：`groups` 到底隔离了哪些通道、什么时候能做残差相加、DenseNet 的通道数怎样增长、轻量化是否真的减少参数、BatchNorm 的 train/eval 状态是否污染线上结果，以及模型制品如何绑定结构和预处理。

所有数据离线合成、固定随机种子、CPU 单线程。微型任务只验证计算图能学习，不等价于 ImageNet 精度复现。

## 1. 两条设计路线与张量合同

MobileNet 用“先逐通道空间卷积、再逐点混合通道”降低计算量。对 $k\times k$ 卷积，普通卷积乘加量近似为

$$HWk^2C_{in}C_{out},$$

depthwise + pointwise 则约为

$$HW(k^2C_{in}+C_{in}C_{out}).$$

DenseNet 采取另一条路线：第 $\ell$ 层接收此前所有特征并只新增 $g$ 个通道，$x_\ell=H_\ell([x_0,\ldots,x_{\ell-1}])$。它改善特征复用，但拼接会增长激活内存。

本册统一输入为 `[N,C,H,W]`，分类 logits 为 `[N,num_classes]`。任何隐式广播、尺寸不匹配和错误通道数都应尽早失败。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from copy import deepcopy  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 340728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.use_deterministic_algorithms(True)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。

## 2. Depthwise separable convolution：`groups` 不是装饰参数

`Conv2d(C,C,k,groups=C)` 把输入拆成 $C$ 个独立组：第 $c$ 个输出只能看到第 $c$ 个输入通道。它**没有通道混合能力**，所以随后必须用 $1\times1$ pointwise convolution 把 $C_{in}$ 映射到 $C_{out}$。

下面不仅检查 shape，还把 grouped convolution 与逐通道调用 `F.conv2d` 的结果逐元素对齐。这个 oracle 能抓住把 `groups` 写成 1、权重索引错位等实现错误。

In [ ]:
class DepthwiseSeparableConv(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels, stride=1):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if in_channels <= 0 or out_channels <= 0 or stride not in (1, 2):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid depthwise-separable configuration")  # 遇到非法合同立即显式失败。
        self.in_channels = int(in_channels)  # 计算并保存当前步骤的中间状态。
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, stride=stride,  # 计算并保存当前步骤的中间状态。
                                   padding=1, groups=in_channels, bias=False)  # 计算并保存当前步骤的中间状态。
        self.depth_bn = nn.BatchNorm2d(in_channels)  # 计算并保存当前步骤的中间状态。
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)  # 计算并保存当前步骤的中间状态。
        self.point_bn = nn.BatchNorm2d(out_channels)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("expected NCHW with configured input channels")  # 遇到非法合同立即显式失败。
        x = F.relu6(self.depth_bn(self.depthwise(x)))  # 计算并保存当前步骤的中间状态。
        return F.relu6(self.point_bn(self.pointwise(x)))  # 返回当前分支计算出的结果。

# groups 数值 oracle：每个输出通道必须只依赖同编号输入通道。
oracle_x = torch.arange(1, 1 + 2 * 4 * 4, dtype=torch.float32).reshape(1, 2, 4, 4)  # 计算并保存当前步骤的中间状态。
oracle_weight = torch.tensor([[[[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]]],  # 计算并保存当前步骤的中间状态。
                              [[[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]]])  # 执行当前语句以推进本节示例。
grouped = F.conv2d(oracle_x, oracle_weight, padding=1, groups=2)  # 计算并保存当前步骤的中间状态。
separate = torch.cat([  # 计算并保存当前步骤的中间状态。
    F.conv2d(oracle_x[:, c:c+1], oracle_weight[c:c+1], padding=1)  # 计算并保存当前步骤的中间状态。
    for c in range(2)  # 遍历输入元素以累积或检查结果。
], dim=1)  # 计算并保存当前步骤的中间状态。
assert torch.equal(grouped, separate)  # 用受控断言验证关键不变量。

depth_probe = DepthwiseSeparableConv(3, 7, stride=2)  # 计算并保存当前步骤的中间状态。
probe_x = torch.randn(2, 3, 16, 16, requires_grad=True)  # 计算并保存当前步骤的中间状态。
probe_y = depth_probe(probe_x)  # 计算并保存当前步骤的中间状态。
probe_y.square().mean().backward()  # 执行当前语句以推进本节示例。
assert probe_y.shape == (2, 7, 8, 8)  # 用受控断言验证关键不变量。
assert depth_probe.depthwise.groups == 3  # 用受控断言验证关键不变量。
assert probe_x.grad is not None and torch.isfinite(probe_x.grad).all()  # 用受控断言验证关键不变量。
assert float(probe_x.grad.norm()) > 0  # 用受控断言验证关键不变量。

## 3. MobileNetV1：宽度、步幅和全局池化

MobileNetV1 的主体重复 depthwise separable block。每次 `stride=2` 同时改变空间分辨率；通道改变只发生在 pointwise 层。分类头使用 adaptive global average pooling，因此不会把某个固定 $H\times W$ 写死在 `Linear` 中。

小型版本保留论文核心算子，但减少 stage 和通道，便于 CPU 教学。`width_mult` 必须同时作用到相邻 block 的输入输出，否则下一层会收到错误通道数。

In [ ]:
class MobileNetV1(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, num_classes=3, width_mult=0.5):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if width_mult <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("width_mult must be positive")  # 遇到非法合同立即显式失败。
        channels = [max(4, int(c * width_mult)) for c in (16, 32, 64, 96)]  # 计算并保存当前步骤的中间状态。
        self.in_channels = int(in_channels)  # 计算并保存当前步骤的中间状态。
        self.stem = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(in_channels, channels[0], 3, padding=1, bias=False),  # 计算并保存当前步骤的中间状态。
            nn.BatchNorm2d(channels[0]), nn.ReLU6(inplace=False),  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        self.features = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            DepthwiseSeparableConv(channels[0], channels[1], stride=2),  # 计算并保存当前步骤的中间状态。
            DepthwiseSeparableConv(channels[1], channels[1], stride=1),  # 计算并保存当前步骤的中间状态。
            DepthwiseSeparableConv(channels[1], channels[2], stride=2),  # 计算并保存当前步骤的中间状态。
            DepthwiseSeparableConv(channels[2], channels[3], stride=1),  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        self.pool = nn.AdaptiveAvgPool2d(1)  # 计算并保存当前步骤的中间状态。
        self.classifier = nn.Linear(channels[3], num_classes)  # 计算并保存当前步骤的中间状态。

    def forward_features(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("MobileNetV1 expected configured NCHW input")  # 遇到非法合同立即显式失败。
        return self.features(self.stem(x))  # 返回当前分支计算出的结果。

    def forward(self, x):  # 定义本节可复用的核心函数。
        features = self.forward_features(x)  # 计算并保存当前步骤的中间状态。
        return self.classifier(self.pool(features).flatten(1))  # 返回当前分支计算出的结果。

mobile_v1_probe = MobileNetV1()  # 计算并保存当前步骤的中间状态。
mobile_v1_features = mobile_v1_probe.forward_features(torch.randn(4, 1, 16, 16))  # 计算并保存当前步骤的中间状态。
mobile_v1_logits = mobile_v1_probe(torch.randn(4, 1, 16, 16))  # 计算并保存当前步骤的中间状态。
assert mobile_v1_features.shape == (4, 48, 4, 4)  # 用受控断言验证关键不变量。
assert mobile_v1_logits.shape == (4, 3)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    mobile_v1_probe(torch.randn(2, 3, 16, 16))  # 执行当前语句以推进本节示例。
    raise AssertionError("channel mismatch must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 4. MobileNetV2 倒残差：先扩张、depthwise、再线性压缩

倒残差块先用 $1\times1$ 把通道扩张 $t$ 倍，再做 depthwise spatial convolution，最后用**不带激活**的线性 pointwise 投影回输出通道。若最后也加 ReLU6，低维瓶颈中的信息更容易被截断。

只有 `stride == 1` 且 `in_channels == out_channels` 时，输入和分支 shape 完全一致，才能逐元素残差相加。下面把残差分支全部置零，直接验证输出必须等于输入；这比只检查 shape 更强。

In [ ]:
class InvertedResidual(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels, stride=1, expansion=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if stride not in (1, 2) or expansion < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid inverted residual configuration")  # 遇到非法合同立即显式失败。
        hidden = int(in_channels * expansion)  # 计算并保存当前步骤的中间状态。
        self.in_channels = int(in_channels)  # 计算并保存当前步骤的中间状态。
        self.use_residual = stride == 1 and in_channels == out_channels  # 计算并保存当前步骤的中间状态。
        layers = []  # 计算并保存当前步骤的中间状态。
        if expansion != 1:  # 按当前条件选择后续控制路径。
            layers += [nn.Conv2d(in_channels, hidden, 1, bias=False),  # 计算并保存当前步骤的中间状态。
                       nn.BatchNorm2d(hidden), nn.ReLU6(inplace=False)]  # 计算并保存当前步骤的中间状态。
        layers += [nn.Conv2d(hidden, hidden, 3, stride=stride, padding=1,  # 计算并保存当前步骤的中间状态。
                             groups=hidden, bias=False),  # 计算并保存当前步骤的中间状态。
                   nn.BatchNorm2d(hidden), nn.ReLU6(inplace=False),  # 计算并保存当前步骤的中间状态。
                   nn.Conv2d(hidden, out_channels, 1, bias=False),  # 计算并保存当前步骤的中间状态。
                   nn.BatchNorm2d(out_channels)]  # 执行当前语句以推进本节示例。
        self.branch = nn.Sequential(*layers)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("inverted residual input mismatch")  # 遇到非法合同立即显式失败。
        branch = self.branch(x)  # 计算并保存当前步骤的中间状态。
        return x + branch if self.use_residual else branch  # 返回当前分支计算出的结果。

class MobileNetV2Tiny(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, num_classes=3):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.in_channels = in_channels  # 计算并保存当前步骤的中间状态。
        self.stem = nn.Sequential(nn.Conv2d(in_channels, 12, 3, padding=1, bias=False),  # 计算并保存当前步骤的中间状态。
                                  nn.BatchNorm2d(12), nn.ReLU6())  # 执行当前语句以推进本节示例。
        self.blocks = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            InvertedResidual(12, 12, 1, 1),  # 执行当前语句以推进本节示例。
            InvertedResidual(12, 20, 2, 2),  # 执行当前语句以推进本节示例。
            InvertedResidual(20, 20, 1, 2),  # 执行当前语句以推进本节示例。
            InvertedResidual(20, 32, 2, 2),  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        self.head = nn.Linear(32, num_classes)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("MobileNetV2Tiny input mismatch")  # 遇到非法合同立即显式失败。
        x = self.blocks(self.stem(x))  # 计算并保存当前步骤的中间状态。
        return self.head(F.adaptive_avg_pool2d(x, 1).flatten(1))  # 返回当前分支计算出的结果。

residual_oracle = InvertedResidual(6, 6, stride=1, expansion=2).eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for parameter in residual_oracle.branch.parameters():  # 遍历输入元素以累积或检查结果。
        parameter.zero_()  # 执行当前语句以推进本节示例。
oracle_input = torch.randn(2, 6, 7, 7)  # 计算并保存当前步骤的中间状态。
assert residual_oracle.use_residual  # 用受控断言验证关键不变量。
assert torch.equal(residual_oracle(oracle_input), oracle_input)  # 用受控断言验证关键不变量。
assert not InvertedResidual(6, 8, stride=1).use_residual  # 用受控断言验证关键不变量。
assert not InvertedResidual(6, 6, stride=2).use_residual  # 用受控断言验证关键不变量。
assert MobileNetV2Tiny()(torch.randn(3, 1, 16, 16)).shape == (3, 3)  # 用受控断言验证关键不变量。

## 5. DenseNet：拼接不是相加

若 block 输入通道为 $C_0$、有 $L$ 层、growth rate 为 $g$，输出通道严格为 $C_0+Lg$。每个 `DenseLayer` 只产生 $g$ 个新特征，然后通过 `torch.cat(..., dim=1)` 保留旧特征。若误写成加法，通道增长和特征复用都会消失。

Transition 使用 $1\times1$ 卷积压缩通道，再以平均池化降低分辨率。教学版使用 BN-ReLU-Conv 的 pre-activation 顺序。

In [ ]:
class DenseLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, growth_rate):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.in_channels = int(in_channels)  # 计算并保存当前步骤的中间状态。
        self.growth_rate = int(growth_rate)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.BatchNorm2d(in_channels)  # 计算并保存当前步骤的中间状态。
        self.conv = nn.Conv2d(in_channels, growth_rate, 3, padding=1, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("DenseLayer input channel mismatch")  # 遇到非法合同立即显式失败。
        new_features = self.conv(F.relu(self.norm(x)))  # 计算并保存当前步骤的中间状态。
        return torch.cat([x, new_features], dim=1)  # 返回当前分支计算出的结果。

class DenseBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, num_layers, growth_rate):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if num_layers < 1 or growth_rate < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid dense block")  # 遇到非法合同立即显式失败。
        layers, channels = [], int(in_channels)  # 计算并保存当前步骤的中间状态。
        for _ in range(num_layers):  # 遍历输入元素以累积或检查结果。
            layers.append(DenseLayer(channels, growth_rate))  # 执行当前语句以推进本节示例。
            channels += growth_rate  # 计算并保存当前步骤的中间状态。
        self.layers = nn.ModuleList(layers)  # 计算并保存当前步骤的中间状态。
        self.out_channels = channels  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        for layer in self.layers:  # 遍历输入元素以累积或检查结果。
            x = layer(x)  # 计算并保存当前步骤的中间状态。
        return x  # 返回当前分支计算出的结果。

class Transition(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if not 0 < out_channels <= in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("transition must not expand channels")  # 遇到非法合同立即显式失败。
        self.norm = nn.BatchNorm2d(in_channels)  # 计算并保存当前步骤的中间状态。
        self.conv = nn.Conv2d(in_channels, out_channels, 1, bias=False)  # 计算并保存当前步骤的中间状态。
        self.pool = nn.AvgPool2d(2, stride=2)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.pool(self.conv(F.relu(self.norm(x))))  # 返回当前分支计算出的结果。

class DenseNetTiny(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, num_classes=3, growth_rate=6):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.in_channels = in_channels  # 计算并保存当前步骤的中间状态。
        self.stem = nn.Conv2d(in_channels, 12, 3, padding=1, bias=False)  # 计算并保存当前步骤的中间状态。
        self.block1 = DenseBlock(12, 3, growth_rate)  # 计算并保存当前步骤的中间状态。
        compressed = self.block1.out_channels // 2  # 计算并保存当前步骤的中间状态。
        self.transition = Transition(self.block1.out_channels, compressed)  # 计算并保存当前步骤的中间状态。
        self.block2 = DenseBlock(compressed, 3, growth_rate)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.BatchNorm2d(self.block2.out_channels)  # 计算并保存当前步骤的中间状态。
        self.classifier = nn.Linear(self.block2.out_channels, num_classes)  # 计算并保存当前步骤的中间状态。

    def forward_features(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("DenseNetTiny input mismatch")  # 遇到非法合同立即显式失败。
        x = self.transition(self.block1(self.stem(x)))  # 计算并保存当前步骤的中间状态。
        return self.block2(x)  # 返回当前分支计算出的结果。

    def forward(self, x):  # 定义本节可复用的核心函数。
        x = F.relu(self.norm(self.forward_features(x)))  # 计算并保存当前步骤的中间状态。
        return self.classifier(F.adaptive_avg_pool2d(x, 1).flatten(1))  # 返回当前分支计算出的结果。

dense_probe = DenseBlock(5, num_layers=4, growth_rate=3)  # 计算并保存当前步骤的中间状态。
dense_output = dense_probe(torch.randn(2, 5, 8, 8, requires_grad=True))  # 计算并保存当前步骤的中间状态。
assert dense_probe.out_channels == 17  # 用受控断言验证关键不变量。
assert dense_output.shape == (2, 17, 8, 8)  # 用受控断言验证关键不变量。
dense_output.mean().backward()  # 执行当前语句以推进本节示例。
assert all(layer.conv.weight.grad is not None for layer in dense_probe.layers)  # 用受控断言验证关键不变量。
dense_net_probe = DenseNetTiny()  # 计算并保存当前步骤的中间状态。
assert dense_net_probe.forward_features(torch.randn(2, 1, 16, 16)).shape == (2, 33, 8, 8)  # 用受控断言验证关键不变量。
assert dense_net_probe(torch.randn(2, 1, 16, 16)).shape == (2, 3)  # 用受控断言验证关键不变量。

## 6. 参数量与理论计算量：先定义比较口径

把一个 $3\times3$ 普通卷积与同尺寸的 depthwise + pointwise 比较，参数量分别为 $9C_{in}C_{out}$ 与 $9C_{in}+C_{in}C_{out}$（此处均无 bias）。比例不包含 BN、激活和内存访问，因此参数少不保证在所有硬件上延迟更低。

端到端模型比较也必须用相同输入通道和类别数；不同宽度、分辨率或分类头会改变结论。

In [ ]:
def parameter_count(module):  # 定义本节可复用的核心函数。
    return sum(parameter.numel() for parameter in module.parameters())  # 返回当前分支计算出的结果。

cin, cout, kernel = 16, 32, 3  # 计算并保存当前步骤的中间状态。
standard = nn.Conv2d(cin, cout, kernel, padding=1, bias=False)  # 计算并保存当前步骤的中间状态。
separable_core = nn.Sequential(  # 计算并保存当前步骤的中间状态。
    nn.Conv2d(cin, cin, kernel, padding=1, groups=cin, bias=False),  # 计算并保存当前步骤的中间状态。
    nn.Conv2d(cin, cout, 1, bias=False),  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
expected_standard = kernel * kernel * cin * cout  # 计算并保存当前步骤的中间状态。
expected_separable = kernel * kernel * cin + cin * cout  # 计算并保存当前步骤的中间状态。
assert parameter_count(standard) == expected_standard  # 用受控断言验证关键不变量。
assert parameter_count(separable_core) == expected_separable  # 用受控断言验证关键不变量。
assert expected_separable < expected_standard / 5  # 用受控断言验证关键不变量。

model_counts = {  # 计算并保存当前步骤的中间状态。
    "MobileNetV1": parameter_count(MobileNetV1()),  # 执行当前语句以推进本节示例。
    "MobileNetV2Tiny": parameter_count(MobileNetV2Tiny()),  # 执行当前语句以推进本节示例。
    "DenseNetTiny": parameter_count(DenseNetTiny()),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
assert all(count > 0 for count in model_counts.values())  # 用受控断言验证关键不变量。
print({"conv_parameter_ratio": expected_separable / expected_standard,  # 执行当前语句以推进本节示例。
       "tiny_model_parameters": model_counts})  # 执行当前语句以推进本节示例。

## 7. 受控学习任务与无泄漏切分

我们合成三类 $16\times16$ 单通道图像：竖条、横条和十字。每张图的位置、宽度和噪声独立变化。先按固定索引生成 train/validation/test，再只用 train 的均值方差做标准化，避免把 validation/test 统计量泄漏进训练。

这是“结构能否反向传播并学会空间模式”的 smoke test。重复纹理比自然图像简单得多，不能据此比较 MobileNet 和 DenseNet 的真实泛化能力。

In [ ]:
def make_pattern_dataset(count, seed):  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    images = torch.zeros(count, 1, 16, 16)  # 计算并保存当前步骤的中间状态。
    labels = torch.arange(count) % 3  # 计算并保存当前步骤的中间状态。
    for index, label in enumerate(labels.tolist()):  # 遍历输入元素以累积或检查结果。
        position = int(torch.randint(4, 12, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
        width = int(torch.randint(1, 3, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
        if label in (0, 2):  # 按当前条件选择后续控制路径。
            images[index, 0, :, position:position + width] = 1.0  # 计算并保存当前步骤的中间状态。
        if label in (1, 2):  # 按当前条件选择后续控制路径。
            images[index, 0, position:position + width, :] = 1.0  # 计算并保存当前步骤的中间状态。
    images += 0.08 * torch.randn(images.shape, generator=generator)  # 计算并保存当前步骤的中间状态。
    return images, labels.long()  # 返回当前分支计算出的结果。

train_raw, train_y = make_pattern_dataset(72, SEED + 1)  # 计算并保存当前步骤的中间状态。
valid_raw, valid_y = make_pattern_dataset(30, SEED + 2)  # 计算并保存当前步骤的中间状态。
test_raw, test_y = make_pattern_dataset(30, SEED + 3)  # 计算并保存当前步骤的中间状态。
train_mean, train_std = train_raw.mean(), train_raw.std().clamp_min(1e-6)  # 计算并保存当前步骤的中间状态。
normalize = lambda tensor: (tensor - train_mean) / train_std  # 计算并保存当前步骤的中间状态。
train_x, valid_x, test_x = map(normalize, (train_raw, valid_raw, test_raw))  # 计算并保存当前步骤的中间状态。

assert train_x.shape == (72, 1, 16, 16)  # 用受控断言验证关键不变量。
assert set(train_y.tolist()) == set(valid_y.tolist()) == set(test_y.tolist()) == {0, 1, 2}  # 用受控断言验证关键不变量。
assert abs(float(train_x.mean())) < 1e-6  # 用受控断言验证关键不变量。
assert not torch.isclose(valid_raw.mean(), train_mean, atol=0, rtol=0)  # 用受控断言验证关键不变量。

## 8. 训练、梯度与 validation checkpoint

为了控制运行时间，训练小型 MobileNetV2。每一步使用完整 train set；validation 只负责选择 checkpoint，test 只在选择完成后读取一次。第一步显式检查所有可训练路径产生有限梯度，训练结束比较受控任务的初末损失和独立 test accuracy。

真实训练应使用 mini-batch、数据增强、学习率调度、混合精度和多次种子，并报告均值与方差；这里故意不把 test 用于调参。

In [ ]:
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
model34 = MobileNetV2Tiny().to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.Adam(model34.parameters(), lr=0.02)  # 计算并保存当前步骤的中间状态。
history, best_validation, best_state = [], float("inf"), None  # 计算并保存当前步骤的中间状态。

for step in range(81):  # 遍历输入元素以累积或检查结果。
    model34.train()  # 执行当前语句以推进本节示例。
    optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits = model34(train_x)  # 计算并保存当前步骤的中间状态。
    loss = F.cross_entropy(logits, train_y)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        first_grad_norm = torch.sqrt(sum(  # 计算并保存当前步骤的中间状态。
            parameter.grad.detach().square().sum()  # 执行当前语句以推进本节示例。
            for parameter in model34.parameters() if parameter.grad is not None  # 遍历输入元素以累积或检查结果。
        ))  # 执行当前语句以推进本节示例。
    optimizer.step()  # 执行当前语句以推进本节示例。
    history.append(float(loss.detach()))  # 执行当前语句以推进本节示例。
    if step % 5 == 0:  # 按当前条件选择后续控制路径。
        model34.eval()  # 执行当前语句以推进本节示例。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            validation_loss = float(F.cross_entropy(model34(valid_x), valid_y))  # 计算并保存当前步骤的中间状态。
        if validation_loss < best_validation:  # 按当前条件选择后续控制路径。
            best_validation = validation_loss  # 计算并保存当前步骤的中间状态。
            best_state = deepcopy(model34.state_dict())  # 计算并保存当前步骤的中间状态。

assert best_state is not None  # 用受控断言验证关键不变量。
model34.load_state_dict(best_state)  # 执行当前语句以推进本节示例。
model34.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    test_logits = model34(test_x)  # 计算并保存当前步骤的中间状态。
test_accuracy = float((test_logits.argmax(1) == test_y).float().mean())  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(first_grad_norm) and float(first_grad_norm) > 0  # 用受控断言验证关键不变量。
assert min(history[-10:]) < history[0] * 0.35  # 用受控断言验证关键不变量。
assert test_accuracy >= 0.90  # 用受控断言验证关键不变量。
print({"train_loss_first_last": [history[0], history[-1]],  # 执行当前语句以推进本节示例。
       "best_validation_loss": best_validation, "test_accuracy": test_accuracy})  # 执行当前语句以推进本节示例。

## 9. BatchNorm 状态合同：推理必须 `eval()`

BatchNorm 在 train 模式用当前 batch 统计量并更新 `running_mean/running_var`；在 eval 模式使用冻结的 running statistics。线上漏掉 `eval()` 会让同一个样本随请求 batch 组成变化，还会悄悄改变模型状态。

下面在模型副本上分别执行 eval 和 train 前向：eval 不得修改 running mean 且重复结果一致，train 则应更新状态。使用副本避免这项探针污染已选择的 checkpoint。

In [ ]:
bn_probe_model = deepcopy(model34)  # 计算并保存当前步骤的中间状态。
first_bn = next(module for module in bn_probe_model.modules() if isinstance(module, nn.BatchNorm2d))  # 计算并保存当前步骤的中间状态。
before_eval = first_bn.running_mean.clone()  # 计算并保存当前步骤的中间状态。
bn_probe_model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    eval_a = bn_probe_model(test_x[:4])  # 计算并保存当前步骤的中间状态。
    eval_b = bn_probe_model(test_x[:4])  # 计算并保存当前步骤的中间状态。
after_eval = first_bn.running_mean.clone()  # 计算并保存当前步骤的中间状态。
assert torch.equal(before_eval, after_eval)  # 用受控断言验证关键不变量。
assert torch.equal(eval_a, eval_b)  # 用受控断言验证关键不变量。

bn_probe_model.train()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    _ = bn_probe_model(torch.full_like(train_x[:8], 4.0))  # 计算并保存当前步骤的中间状态。
after_train = first_bn.running_mean.clone()  # 计算并保存当前步骤的中间状态。
assert not torch.equal(after_eval, after_train)  # 用受控断言验证关键不变量。

# 训练模式下，单样本输出依赖同 batch 的其他样本；eval 不应如此。
bn_probe_model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    alone = bn_probe_model(test_x[:1])  # 计算并保存当前步骤的中间状态。
    in_batch = bn_probe_model(torch.cat([test_x[:1], test_x[1:8]], dim=0))[:1]  # 计算并保存当前步骤的中间状态。
assert torch.allclose(alone, in_batch, atol=1e-6)  # 用受控断言验证关键不变量。

## 10. 制品合同：发布者信任锚、训练快照与可重建预处理

制品自己保存 `manifest_sha256` 只能发现意外损坏：攻击者若整体替换权重、manifest 和内部摘要，它们仍然“自洽”。本节把发布者侧只读登记表 `artifact_id/version -> expected bundle digest` 当作 package 外的信任锚。bundle digest 同时覆盖 canonical manifest 与 state 中每个 tensor 的 `key/dtype/shape/bytes`；内部 hash 全部重签也不能改变发布登记值。

语义校验同样不能停在“能 strict load”：类别必须唯一非空且 `num_classes == len(class_names)`，config、输入 shape、state schema 和标准化 recipe 都要精确匹配。train/validation/test 的原始图像、target、count 与 seed 被绑定为 split snapshot；因为这里的数据生成受控，loader 会重新生成 train split，并只由这个绑定快照重算 mean/std。生产中信任锚应来自签名发布元数据、透明日志或只读制品服务，而不是和模型一起放在可替换目录。

In [ ]:
def canonical_json34(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, sort_keys=True, separators=(",", ":"),  # 返回当前分支计算出的结果。
                      ensure_ascii=False).encode("utf-8")  # 计算并保存当前步骤的中间状态。

def _feed_digest34(hasher, payload):  # 定义本节可复用的核心函数。
    hasher.update(len(payload).to_bytes(8, "big"))  # 执行当前语句以推进本节示例。
    hasher.update(payload)  # 执行当前语句以推进本节示例。

def clone_state34(state_dict):  # 定义本节可复用的核心函数。
    if not hasattr(state_dict, "items"):  # 按当前条件选择后续控制路径。
        raise ValueError("state_dict must be a mapping")  # 遇到非法合同立即显式失败。
    cloned = {}  # 计算并保存当前步骤的中间状态。
    for key, tensor in state_dict.items():  # 遍历输入元素以累积或检查结果。
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise ValueError("state entries must be string -> Tensor")  # 遇到非法合同立即显式失败。
        cloned[key] = tensor.detach().cpu().contiguous().clone()  # 计算并保存当前步骤的中间状态。
    return cloned  # 返回当前分支计算出的结果。

def _update_state_digest34(hasher, state_dict):  # 定义本节可复用的核心函数。
    if not isinstance(state_dict, dict) or not state_dict:  # 按当前条件选择后续控制路径。
        raise ValueError("artifact state_dict must be a non-empty plain dict")  # 遇到非法合同立即显式失败。
    for key in sorted(state_dict):  # 遍历输入元素以累积或检查结果。
        tensor = state_dict[key]  # 计算并保存当前步骤的中间状态。
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid state entry")  # 遇到非法合同立即显式失败。
        cpu = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        header = canonical_json34({"key": key, "dtype": str(cpu.dtype),  # 计算并保存当前步骤的中间状态。
                                   "shape": list(cpu.shape)})  # 执行当前语句以推进本节示例。
        raw = cpu.reshape(-1).view(torch.uint8).numpy().tobytes()  # 计算并保存当前步骤的中间状态。
        _feed_digest34(hasher, header)  # 执行当前语句以推进本节示例。
        _feed_digest34(hasher, raw)  # 执行当前语句以推进本节示例。

def canonical_state_digest34(state_dict):  # 定义本节可复用的核心函数。
    hasher = sha256()  # 计算并保存当前步骤的中间状态。
    _feed_digest34(hasher, b"canonical-state-dict-v1")  # 执行当前语句以推进本节示例。
    _update_state_digest34(hasher, state_dict)  # 执行当前语句以推进本节示例。
    return hasher.hexdigest()  # 返回当前分支计算出的结果。

def canonical_bundle_digest34(manifest, state_dict):  # 定义本节可复用的核心函数。
    hasher = sha256()  # 计算并保存当前步骤的中间状态。
    _feed_digest34(hasher, b"canonical-model-bundle-v1")  # 执行当前语句以推进本节示例。
    _feed_digest34(hasher, canonical_json34(manifest))  # 执行当前语句以推进本节示例。
    _update_state_digest34(hasher, state_dict)  # 执行当前语句以推进本节示例。
    return hasher.hexdigest()  # 返回当前分支计算出的结果。

def state_schema34(state_dict):  # 定义本节可复用的核心函数。
    return [{"key": key, "dtype": str(state_dict[key].dtype),  # 返回当前分支计算出的结果。
             "shape": list(state_dict[key].shape)} for key in sorted(state_dict)]  # 执行当前语句以推进本节示例。

def expected_pattern_release34():  # 定义本节可复用的核心函数。
    split_specs = {"train": (72, SEED + 1), "validation": (30, SEED + 2),  # 计算并保存当前步骤的中间状态。
                   "test": (30, SEED + 3)}  # 执行当前语句以推进本节示例。
    splits, rebuilt = {}, {}  # 计算并保存当前步骤的中间状态。
    for name, (count, seed) in split_specs.items():  # 遍历输入元素以累积或检查结果。
        images, labels = make_pattern_dataset(count, seed)  # 计算并保存当前步骤的中间状态。
        rebuilt[name] = (images, labels)  # 计算并保存当前步骤的中间状态。
        snapshot_state = clone_state34({"images": images, "labels": labels})  # 计算并保存当前步骤的中间状态。
        splits[name] = {"count": count, "seed": seed,  # 计算并保存当前步骤的中间状态。
                        "images_labels_sha256": canonical_state_digest34(snapshot_state)}  # 执行当前语句以推进本节示例。
    train_images = rebuilt["train"][0]  # 计算并保存当前步骤的中间状态。
    std_floor = 1e-6  # 计算并保存当前步骤的中间状态。
    derived_mean = float(train_images.mean())  # 计算并保存当前步骤的中间状态。
    derived_std = float(train_images.std().clamp_min(std_floor))  # 计算并保存当前步骤的中间状态。
    data_contract = {"dataset_recipe": "controlled-bars-v1", "splits": splits,  # 计算并保存当前步骤的中间状态。
                     "target_schema": "int64 class index"}  # 执行当前语句以推进本节示例。
    normalizer = {"kind": "standard-score", "fit_split": "train",  # 计算并保存当前步骤的中间状态。
                  "mean": derived_mean, "std": derived_std, "std_floor": std_floor}  # 执行当前语句以推进本节示例。
    return data_contract, normalizer  # 返回当前分支计算出的结果。

EXPECTED_CONFIG34 = {"in_channels": 1, "num_classes": 3}  # 计算并保存当前步骤的中间状态。
EXPECTED_CLASSES34 = ["vertical", "horizontal", "cross"]  # 计算并保存当前步骤的中间状态。
EXPECTED_INPUT_SHAPE34 = [1, 16, 16]  # 计算并保存当前步骤的中间状态。
EXPECTED_PREPROCESS34 = {"recipe": "standardize-from-bound-train-v1",  # 计算并保存当前步骤的中间状态。
                         "formula": "(x-mean)/max(std,std_floor)",  # 执行当前语句以推进本节示例。
                         "layout": "NCHW", "dtype": "float32"}  # 执行当前语句以推进本节示例。
ARTIFACT_ID34, ARTIFACT_VERSION34 = "vision.controlled-mobilenetv2", "1.0.0"  # 计算并保存当前步骤的中间状态。

def build_artifact(model, state_dict):  # 定义本节可复用的核心函数。
    if type(model) is not MobileNetV2Tiny:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher only accepts the audited MobileNetV2Tiny class")  # 遇到非法合同立即显式失败。
    config = {"in_channels": int(model.in_channels),  # 计算并保存当前步骤的中间状态。
              "num_classes": int(model.head.out_features)}  # 执行当前语句以推进本节示例。
    state = clone_state34(state_dict)  # 计算并保存当前步骤的中间状态。
    probe = MobileNetV2Tiny(**config)  # 计算并保存当前步骤的中间状态。
    probe.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    data_contract, normalizer = expected_pattern_release34()  # 计算并保存当前步骤的中间状态。
    manifest = {  # 计算并保存当前步骤的中间状态。
        "schema_version": 2, "artifact_id": ARTIFACT_ID34,  # 执行当前语句以推进本节示例。
        "artifact_version": ARTIFACT_VERSION34, "architecture": "MobileNetV2Tiny",  # 执行当前语句以推进本节示例。
        "architecture_config": config, "model_state_schema": state_schema34(state),  # 执行当前语句以推进本节示例。
        "class_names": list(EXPECTED_CLASSES34), "input_shape": list(EXPECTED_INPUT_SHAPE34),  # 执行当前语句以推进本节示例。
        "preprocess_recipe": deepcopy(EXPECTED_PREPROCESS34), "normalizer": normalizer,  # 执行当前语句以推进本节示例。
        "data_contract": data_contract, "torch_version": torch.__version__,  # 执行当前语句以推进本节示例。
        "state_digest_sha256": canonical_state_digest34(state),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return {"manifest": manifest,  # 返回当前分支计算出的结果。
            "manifest_sha256": sha256(canonical_json34(manifest)).hexdigest(),  # 执行当前语句以推进本节示例。
            "bundle_sha256": canonical_bundle_digest34(manifest, state),  # 执行当前语句以推进本节示例。
            "state_dict": state}  # 执行当前语句以推进本节示例。

def validate_mobile_contract34(manifest, state_dict):  # 定义本节可复用的核心函数。
    required = {"schema_version", "artifact_id", "artifact_version", "architecture",  # 计算并保存当前步骤的中间状态。
                "architecture_config", "model_state_schema", "class_names", "input_shape",  # 执行当前语句以推进本节示例。
                "preprocess_recipe", "normalizer", "data_contract", "torch_version",  # 执行当前语句以推进本节示例。
                "state_digest_sha256"}  # 执行当前语句以推进本节示例。
    if set(manifest) != required or manifest["schema_version"] != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest schema mismatch")  # 遇到非法合同立即显式失败。
    if (manifest["artifact_id"], manifest["artifact_version"]) != (ARTIFACT_ID34, ARTIFACT_VERSION34):  # 按当前条件选择后续控制路径。
        raise ValueError("artifact identity mismatch")  # 遇到非法合同立即显式失败。
    if (manifest["architecture"] != "MobileNetV2Tiny" or  # 按当前条件选择后续控制路径。
            manifest["architecture_config"] != EXPECTED_CONFIG34):  # 计算并保存当前步骤的中间状态。
        raise ValueError("architecture/config mismatch")  # 遇到非法合同立即显式失败。
    names = manifest["class_names"]  # 计算并保存当前步骤的中间状态。
    if (names != EXPECTED_CLASSES34 or len(names) != manifest["architecture_config"]["num_classes"] or  # 按当前条件选择后续控制路径。
            len(set(names)) != len(names) or any(not isinstance(x, str) or not x.strip() for x in names)):  # 计算并保存当前步骤的中间状态。
        raise ValueError("class mapping must be exact, unique and non-empty")  # 遇到非法合同立即显式失败。
    if manifest["input_shape"] != EXPECTED_INPUT_SHAPE34:  # 按当前条件选择后续控制路径。
        raise ValueError("input shape mismatch")  # 遇到非法合同立即显式失败。
    if manifest["preprocess_recipe"] != EXPECTED_PREPROCESS34:  # 按当前条件选择后续控制路径。
        raise ValueError("preprocess recipe mismatch")  # 遇到非法合同立即显式失败。
    expected_data, expected_normalizer = expected_pattern_release34()  # 计算并保存当前步骤的中间状态。
    if manifest["data_contract"] != expected_data:  # 按当前条件选择后续控制路径。
        raise ValueError("train/target/split snapshot mismatch")  # 遇到非法合同立即显式失败。
    normalizer = manifest["normalizer"]  # 计算并保存当前步骤的中间状态。
    if (normalizer != expected_normalizer or not math.isfinite(normalizer["mean"]) or  # 按当前条件选择后续控制路径。
            not math.isfinite(normalizer["std"]) or normalizer["std"] <= 0):  # 计算并保存当前步骤的中间状态。
        raise ValueError("normalizer is not derived from the bound train snapshot")  # 遇到非法合同立即显式失败。
    expected_schema = state_schema34(MobileNetV2Tiny(**EXPECTED_CONFIG34).state_dict())  # 计算并保存当前步骤的中间状态。
    if manifest["model_state_schema"] != expected_schema or state_schema34(state_dict) != expected_schema:  # 按当前条件选择后续控制路径。
        raise ValueError("model state schema mismatch")  # 遇到非法合同立即显式失败。

def load_trusted_mobile(artifact):  # 定义本节可复用的核心函数。
    if not isinstance(artifact, dict) or set(artifact) != {  # 按当前条件选择后续控制路径。
            "manifest", "manifest_sha256", "bundle_sha256", "state_dict"}:  # 执行当前语句以推进本节示例。
        raise ValueError("artifact package schema mismatch")  # 遇到非法合同立即显式失败。
    manifest, state = artifact["manifest"], artifact["state_dict"]  # 计算并保存当前步骤的中间状态。
    if not isinstance(manifest, dict):  # 按当前条件选择后续控制路径。
        raise ValueError("manifest must be a dict")  # 遇到非法合同立即显式失败。
    actual_bundle = canonical_bundle_digest34(manifest, state)  # 计算并保存当前步骤的中间状态。
    identity = (manifest.get("artifact_id"), manifest.get("artifact_version"))  # 计算并保存当前步骤的中间状态。
    expected_bundle = PUBLISHER_REGISTRY34.get(identity)  # 计算并保存当前步骤的中间状态。
    if expected_bundle is None or actual_bundle != expected_bundle:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher registry rejected this bundle")  # 遇到非法合同立即显式失败。
    if artifact["bundle_sha256"] != actual_bundle:  # 按当前条件选择后续控制路径。
        raise ValueError("internal bundle digest mismatch")  # 遇到非法合同立即显式失败。
    if sha256(canonical_json34(manifest)).hexdigest() != artifact["manifest_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest digest mismatch")  # 遇到非法合同立即显式失败。
    if canonical_state_digest34(state) != manifest["state_digest_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("canonical state digest mismatch")  # 遇到非法合同立即显式失败。
    validate_mobile_contract34(manifest, state)  # 执行当前语句以推进本节示例。
    loaded = MobileNetV2Tiny(**manifest["architecture_config"])  # 计算并保存当前步骤的中间状态。
    loaded.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    return loaded.eval()  # 返回当前分支计算出的结果。

def resign_inside34(artifact):  # 定义本节可复用的核心函数。
    artifact["manifest"]["state_digest_sha256"] = canonical_state_digest34(artifact["state_dict"])  # 计算并保存当前步骤的中间状态。
    artifact["manifest_sha256"] = sha256(canonical_json34(artifact["manifest"])).hexdigest()  # 计算并保存当前步骤的中间状态。
    artifact["bundle_sha256"] = canonical_bundle_digest34(artifact["manifest"], artifact["state_dict"])  # 计算并保存当前步骤的中间状态。
    return artifact  # 返回当前分支计算出的结果。

artifact34 = build_artifact(model34, model34.state_dict())  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY34 = MappingProxyType({  # 计算并保存当前步骤的中间状态。
    (ARTIFACT_ID34, ARTIFACT_VERSION34): artifact34["bundle_sha256"]  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。
loaded34 = load_trusted_mobile(artifact34)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    assert torch.equal(loaded34(test_x[:5]), model34(test_x[:5]))  # 用受控断言验证关键不变量。

# 整体替换为四分类模型，并重签 package 内全部摘要；外部发布登记仍拒绝。
forged_model34 = MobileNetV2Tiny(num_classes=4)  # 计算并保存当前步骤的中间状态。
forged_four_class34 = deepcopy(artifact34)  # 计算并保存当前步骤的中间状态。
forged_four_class34["state_dict"] = clone_state34(forged_model34.state_dict())  # 计算并保存当前步骤的中间状态。
forged_four_class34["manifest"]["architecture_config"]["num_classes"] = 4  # 计算并保存当前步骤的中间状态。
forged_four_class34["manifest"]["class_names"] = ["vertical", "horizontal", "cross", "other"]  # 计算并保存当前步骤的中间状态。
forged_four_class34["manifest"]["model_state_schema"] = state_schema34(forged_four_class34["state_dict"])  # 计算并保存当前步骤的中间状态。
resign_inside34(forged_four_class34)  # 执行当前语句以推进本节示例。
assert forged_four_class34["bundle_sha256"] == canonical_bundle_digest34(  # 用受控断言验证关键不变量。
    forged_four_class34["manifest"], forged_four_class34["state_dict"])  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    load_trusted_mobile(forged_four_class34)  # 执行当前语句以推进本节示例。
    raise AssertionError("self-signed four-class replacement must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

# 权重不变但把 train-only mean 平移 7，同样重签内部摘要，仍不能越过发布信任锚。
forged_mean34 = deepcopy(artifact34)  # 计算并保存当前步骤的中间状态。
forged_mean34["manifest"]["normalizer"]["mean"] += 7.0  # 计算并保存当前步骤的中间状态。
resign_inside34(forged_mean34)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    load_trusted_mobile(forged_mean34)  # 执行当前语句以推进本节示例。
    raise AssertionError("self-signed normalizer replacement must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

try:  # 尝试执行可能失败的受控操作。
    PUBLISHER_REGISTRY34[(ARTIFACT_ID34, ARTIFACT_VERSION34)] = forged_mean34["bundle_sha256"]  # 计算并保存当前步骤的中间状态。
    raise AssertionError("publisher registry must be immutable")  # 遇到非法合同立即显式失败。
except TypeError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 11. 常见失败模式、生产差距与复现清单

1. **depthwise 后漏掉 pointwise**：空间特征更新了，但不同通道永远不能交流。
2. **倒残差无条件相加**：stride 或通道改变时不是残差连接，应直接拒绝或使用显式 projection；MobileNetV2 原块选择前者。
3. **DenseNet 写成相加**：growth-rate 合同被破坏；应逐层断言通道为 $C_0+\ell g$。
4. **参数少等同于延迟低**：depthwise 算子可能受内存访问、kernel 实现和设备支持限制；必须在目标硬件做 warm-up 后测 p50/p95/p99。
5. **BN 探针污染模型**：所有校准/诊断在副本上做；小 batch 可评估 GroupNorm、SyncBN 或冻结 BN。
6. **只保存权重或自签 hash**：必须用发布者外部信任锚绑定权重、数据快照、预处理、标签和结构。

生产复现还需真实数据治理、分组切分、增强策略、蒸馏/量化校准、AMP 数值审计、多随机种子、吞吐与峰值内存、OOD/鲁棒性/公平性测试。导出 ONNX 或移动端格式后必须逐层或端到端对齐数值，不能只看导出成功。

### 论文来源

- Howard et al., [*MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications*](https://arxiv.org/abs/1704.04861), 2017.
- Sandler et al., [*MobileNetV2: Inverted Residuals and Linear Bottlenecks*](https://arxiv.org/abs/1801.04381), 2018.
- Huang et al., [*Densely Connected Convolutional Networks*](https://arxiv.org/abs/1608.06993), CVPR 2017.

本册复现核心计算图和工程合同；没有声称复现论文训练配方、数据规模或榜单结果。